# Email Spam Detection using Machine Learning

## Objective
To classify messages as Spam or Ham using Machine Learning.

## Algorithms
1. Multinomial Naive Bayes
2. Logistic Regression

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
print("Class counts:")
print(df["label"].value_counts())

print("\nClass percentages:")
print(
    (df["label"].value_counts(normalize=True) * 100).round(2)
)

Class counts:
label
ham     4825
spam     747
Name: count, dtype: int64

Class percentages:
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64


In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nLabel values:")
print(df["label"].value_counts())

Missing values:
label      0
message    0
dtype: int64

Duplicate rows: 403

Label values:
label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (5169, 2)


In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)

df[["message", "clean_message", "label"]].head()

,message,clean_message,label
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,ham
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...,spam
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,ham
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,ham


In [7]:
print("Original message:")
print(df["message"].iloc[0])

print("\nCleaned message:")
print(df["clean_message"].iloc[0])

Original message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Cleaned message:
go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = tfidf.fit_transform(df["clean_message"])

print("TF-IDF transformation completed!")
print("Feature matrix shape:", X.shape)

TF-IDF transformation completed!
Feature matrix shape: (5169, 5000)


In [9]:
feature_names = tfidf.get_feature_names_out()

print("First 20 words:")
print(feature_names[:20])

First 20 words:
['aa' 'aah' 'abi' 'ability' 'abiola' 'abj' 'able' 'abt' 'abta' 'aburo'
 'ac' 'academic' 'acc' 'accept' 'access' 'accident' 'accidentally'
 'accordingly' 'account' 'accounts']


In [10]:
y = df["label"].map({"ham": 0, "spam": 1})

print(y.value_counts())
print("\nTarget variable created successfully!")

label
0    4516
1     653
Name: count, dtype: int64

Target variable created successfully!


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining labels:", len(y_train))
print("Testing labels:", len(y_test))

Training data shape: (4135, 5000)
Testing data shape: (1034, 5000)

Training labels: 4135
Testing labels: 1034


In [12]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [13]:
y_pred_nb = nb_model.predict(X_test)

print("Predictions completed!")
print("First 10 predictions:", y_pred_nb[:10])

Predictions completed!
First 10 predictions: [0 0 0 0 0 0 0 0 0 0]


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred_nb)
precision = precision_score(y_test, y_pred_nb, zero_division=0)
recall = recall_score(y_test, y_pred_nb, zero_division=0)
f1 = f1_score(y_test, y_pred_nb, zero_division=0)

print("Naive Bayes Model Evaluation")
print("----------------------------")
print("Accuracy :", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall   :", round(recall * 100, 2), "%")
print("F1-Score :", round(f1 * 100, 2), "%")

Naive Bayes Model Evaluation
----------------------------
Accuracy : 97.2 %
Precision: 100.0 %
Recall   : 77.86 %
F1-Score : 87.55 %


In [15]:
print(classification_report(
    y_test,
    y_pred_nb,
    labels=[0, 1],
    target_names=["Ham", "Spam"],
    zero_division=0
))

              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       903
        Spam       1.00      0.78      0.88       131

    accuracy                           0.97      1034
   macro avg       0.98      0.89      0.93      1034
weighted avg       0.97      0.97      0.97      1034



In [16]:
cm = confusion_matrix(y_test, y_pred_nb, labels=[0, 1])

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[903   0]
 [ 29 102]]


In [17]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)

print("Logistic Regression Evaluation")
print("------------------------------")
print("Accuracy :", round(accuracy_lr * 100, 2), "%")
print("Precision:", round(precision_lr * 100, 2), "%")
print("Recall   :", round(recall_lr * 100, 2), "%")
print("F1-Score :", round(f1_lr * 100, 2), "%")

Logistic Regression Evaluation
------------------------------
Accuracy : 94.39 %
Precision: 97.4 %
Recall   : 57.25 %
F1-Score : 72.12 %


In [19]:
results = pd.DataFrame({
    "Model": ["Multinomial Naive Bayes", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr)
    ],
    "Precision": [
        precision_score(y_test, y_pred_nb, zero_division=0),
        precision_score(y_test, y_pred_lr, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_nb, zero_division=0),
        recall_score(y_test, y_pred_lr, zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_nb, zero_division=0),
        f1_score(y_test, y_pred_lr, zero_division=0)
    ]
})

results[["Accuracy", "Precision", "Recall", "F1-Score"]] = (
    results[["Accuracy", "Precision", "Recall", "F1-Score"]].round(4)
)

results

,Model,Accuracy,Precision,Recall,F1-Score
0,Multinomial Naive Bayes,0.9720,1.000,0.7786,0.8755
1,Logistic Regression,0.9439,0.974,0.5725,0.7212


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)

print("Logistic Regression Evaluation")
print("------------------------------")
print("Accuracy :", round(accuracy_lr * 100, 2), "%")
print("Precision:", round(precision_lr * 100, 2), "%")
print("Recall   :", round(recall_lr * 100, 2), "%")
print("F1-Score :", round(f1_lr * 100, 2), "%")

Logistic Regression Evaluation
------------------------------
Accuracy : 94.39 %
Precision: 97.4 %
Recall   : 57.25 %
F1-Score : 72.12 %


In [21]:
results = pd.DataFrame({
    "Model": ["Multinomial Naive Bayes", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr)
    ],
    "Precision": [
        precision_score(y_test, y_pred_nb, zero_division=0),
        precision_score(y_test, y_pred_lr, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_nb, zero_division=0),
        recall_score(y_test, y_pred_lr, zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_nb, zero_division=0),
        f1_score(y_test, y_pred_lr, zero_division=0)
    ]
})

results[["Accuracy", "Precision", "Recall", "F1-Score"]] = (
    results[["Accuracy", "Precision", "Recall", "F1-Score"]].round(4)
)

results

,Model,Accuracy,Precision,Recall,F1-Score
0,Multinomial Naive Bayes,0.9720,1.000,0.7786,0.8755
1,Logistic Regression,0.9439,0.974,0.5725,0.7212


In [22]:
def predict_message(message):
    cleaned_message = clean_text(message)
    message_vector = tfidf.transform([cleaned_message])

    prediction = lr_model.predict(message_vector)[0]

    if prediction == 1:
        return "Spam Message"
    else:
        return "Ham (Not Spam)"

test_message = "Congratulations! You have won a free prize. Call now!"

print("Message:", test_message)
print("Prediction:", predict_message(test_message))

Message: Congratulations! You have won a free prize. Call now!
Prediction: Spam Message


## Conclusion

In this project, we developed an SMS spam detection system
using Machine Learning.

We performed text preprocessing and converted text into
numerical features using TF-IDF.

We trained and evaluated two classifiers:
1. Multinomial Naive Bayes
2. Logistic Regression

We compared both models using Accuracy, Precision, Recall,
and F1-score.

The results help us understand how effectively each model
distinguishes spam messages from legitimate messages.